In [1]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder,OrdinalEncoder

In [2]:

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/blastchar/telco-customer-churn


In [3]:
df=pd.read_csv(f'{path}/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [4]:
X= df.drop('Churn',axis=1)
Y= df["Churn"]

In [5]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Feature Engineering

## Feature 1 : ancienneté en mois

In [6]:
# Segmentation ancienneté
df['tenure_segment'] = pd.cut(df['tenure'], bins=[0, 12, 24, 60, 100],
                               labels=['<1yr', '1-2yr', '2-5yr', '>5yr'])

## Feature 2 : prix moyen par mois d'ancienneté

In [7]:
df["avg_monthly_spend"] = (
    df["TotalCharges"] /
    (df["tenure"] + 1)
)

## Feature 3 : nombre de services utilisés

In [8]:
services = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["service_count"] = (
    df[services]
    .replace({"Yes":1,"No":0,"No internet service":0})
    .sum(axis=1)
)

/tmp/ipykernel_16/1418318358.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"Yes":1,"No":0,"No internet service":0})


In [9]:
df["risk_score"] = 0

df.loc[df["Contract"]=="Month-to-month",
       "risk_score"] += 1

df.loc[df["TechSupport"]=="No",
       "risk_score"] += 1

df.loc[df["OnlineSecurity"]=="No",
       "risk_score"] += 1

df.loc[df["tenure"]<12,
       "risk_score"] += 1

In [10]:
df.to_csv(
    "churn_processed.csv",
    index=False
)

In [11]:
data_processed= pd.read_csv('/kaggle/working/churn_processed.csv')
data_processed

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_segment,avg_monthly_spend,service_count,risk_score
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,<1yr,14.925000,1,4
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,One year,No,Mailed check,56.95,1889.50,No,2-5yr,53.985714,2,1
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,<1yr,36.050000,2,3
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,One year,No,Bank transfer (automatic),42.30,1840.75,No,2-5yr,40.016304,3,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,<1yr,50.550000,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,One year,Yes,Mailed check,84.80,1990.50,No,1-2yr,79.620000,5,0
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,One year,Yes,Credit card (automatic),103.20,7362.90,No,>5yr,100.861644,4,2
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,Month-to-month,Yes,Electronic check,29.60,346.45,No,<1yr,28.870833,1,3
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,Month-to-month,Yes,Mailed check,74.40,306.60,Yes,<1yr,61.320000,0,4


In [12]:
df.groupby("Churn")[
    [
        "avg_monthly_spend",
        "service_count",
        "risk_score"
    ]
].mean()

,avg_monthly_spend,service_count,risk_score
Churn,,,
No,57.779786,2.135292,1.421337
Yes,62.683301,1.768325,2.975388


In [13]:
pd.crosstab(
    df["risk_score"],
    df["Churn"],
    normalize="index"
)

Churn,No,Yes
risk_score,,
0,0.964952,0.035048
1,0.904025,0.095975
2,0.754935,0.245065
3,0.579371,0.420629
4,0.347295,0.652705
